# Fit PCA

> Fit PCA on pre-encoded embeddings and save reduced representations for generative model training.

In [ ]:
#| default_exp fit_pca

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import torch
import pickle
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from sklearn.decomposition import PCA
from omegaconf import DictConfig
import hydra

In [ ]:
#| export
def load_level_embeddings(encoded_dir: Path, split: str, level: int = 0, key: str = 'emb2'):
    """Load all patch embeddings at a given hierarchy level from pre-encoded chunk files.
    Returns (N_total * N_patches, D) — all patches flattened for PCA fitting.
    key: 'emb1', 'emb2', or 'emb3' (which image from the triplet)."""
    chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
    assert chunks, f"No chunks found for split={split} in {encoded_dir}"
    all_embs = []
    for chunk_path in tqdm(chunks, desc=f"Loading {split} L{level}"):
        chunk = torch.load(chunk_path, weights_only=False)
        for batch_rec in chunk:
            emb = batch_rec[key][level]          # (B, N_patches, D)
            B, P, D = emb.shape
            all_embs.append(emb.reshape(B * P, D))   # flatten patches
    return torch.cat(all_embs, dim=0).float().numpy()  # (N_total * N_patches, D)

In [ ]:
#| export
def fit_and_save_pca(encoded_dir: str, output_dir: str, levels: list = None,
                     n_components: int = 20, key: str = 'emb2',
                     fine_levels: list = None, fine_n_components = None):
    """Fit PCA on per-patch training embeddings for each level, then project and save per-chunk.

    PCA is fit on (N_total * N_patches, D) — all patches from all samples, preserving spatial variance.
    Levels where D <= n_components skip PCA (identity) to avoid fitting on hundreds of millions of rows.

    fine_n_components: int (same for all fine levels) or list (one per fine level).
      e.g. fine_n_components=[4, 3] for L4=4 components, L5=3 components.

    Coarse output: <output_dir>/{split}_chunk{idx}_pca{n_components}.pt   — keys 'L0', 'L1', ...
    Fine output:   <output_dir>/{split}_chunk{idx}_fine_pca{suffix}.pt    — keys 'L4', 'L5', ...
      where suffix is e.g. '4_3' for [4,3] or '4' for 4.
    """
    encoded_dir     = Path(encoded_dir)
    output_dir      = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    fine_levels_set = set(fine_levels) if fine_levels else set()

    if levels is None:
        sample = torch.load(sorted(encoded_dir.glob("train_chunk*.pt"))[0], weights_only=False)
        levels = list(range(len(sample[0][key])))

    coarse_levels = [l for l in levels if l not in fine_levels_set]
    active_fine   = [l for l in levels if l in fine_levels_set]

    # Normalize fine_n_components to per-level list and build file suffix
    if active_fine and fine_n_components is not None:
        if isinstance(fine_n_components, int):
            fn_list = [fine_n_components] * len(active_fine)
            fn_suffix = f"fine_pca{fine_n_components}"
        else:
            fn_list = list(fine_n_components)
            fn_suffix = "fine_pca" + "_".join(str(n) for n in fn_list)
    else:
        fn_list, fn_suffix = [], None

    def _fit_and_project(level_list, n_comp_list, file_suffix):
        """Fit PCA on level_list; save PKLs + per-chunk files."""
        if not level_list:
            return {}
        pcas = {}
        for level, n_comp in zip(level_list, n_comp_list):
            sample = torch.load(sorted(encoded_dir.glob("train_chunk*.pt"))[0], weights_only=False)
            D = sample[0][key][level].shape[-1]
            n = min(n_comp, D)
            if D <= n_comp:
                print(f"L{level}: D={D} <= n_components={n_comp}, saving raw")
                pcas[level] = None
                continue
            print(f"Fitting PCA(n={n}) on L{level} ...")
            train_emb = load_level_embeddings(encoded_dir, "train", level=level, key=key)
            print(f"  shape: {train_emb.shape}")
            pca = PCA(n_components=n, whiten=False)
            pca.fit(train_emb)
            var = pca.explained_variance_ratio_.cumsum()[-1]
            print(f"  variance explained: {var:.1%}")
            with open(output_dir / f"pca_L{level}_n{n_comp}.pkl", "wb") as f:
                pickle.dump(pca, f)
            pcas[level] = pca
            del train_emb

        for split in ["train", "val"]:
            chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
            print(f"\nProjecting {len(chunks)} {split} chunks [{file_suffix}]...")
            for chunk_path in tqdm(chunks, desc=split):
                data = torch.load(chunk_path, weights_only=False)
                out = {}
                for level, n_comp in zip(level_list, n_comp_list):
                    embs = torch.cat([rec[key][level].float() for rec in data], dim=0)
                    N, P, D = embs.shape
                    if pcas[level] is None:
                        out[f"L{level}"] = embs
                    else:
                        flat = embs.reshape(N * P, D).numpy()
                        proj = torch.tensor(pcas[level].transform(flat),
                                            dtype=torch.float32).reshape(N, P, -1)
                        out[f"L{level}"] = proj
                torch.save(out, output_dir / f"{chunk_path.stem}_{file_suffix}.pt")
            print(f"  done → {output_dir}")
        return pcas

    coarse_pcas = _fit_and_project(coarse_levels,
                                    [n_components] * len(coarse_levels),
                                    f"pca{n_components}")
    fine_pcas   = (_fit_and_project(active_fine, fn_list, fn_suffix)
                   if active_fine else {})
    return coarse_pcas, fine_pcas

## Interactive usage

In [ ]:
#| eval: false
# Example: fit PCA(32) on L0 embeddings
pca = fit_and_save_pca(
    encoded_dir='~/datasets/POP909_encoded',
    output_dir='~/datasets/POP909_pca',
    level=0,
    n_components=32,
)

In [ ]:
#| export
#| eval: false
@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def fit_pca_main(cfg: DictConfig):
    fc = cfg.get('fitpca', {})
    encoded_dir       = os.path.expandvars(os.path.expanduser(str(fc.get('encoded_dir', cfg.preencode.output_dir))))
    output_dir        = os.path.expandvars(os.path.expanduser(str(fc.get('output_dir', encoded_dir + '_pca'))))
    levels            = list(fc.levels) if fc.get('levels') else None
    n_components      = int(fc.get('n_components', 20))
    key               = str(fc.get('key', 'emb2'))
    fine_levels       = list(fc.fine_levels) if fc.get('fine_levels') else None
    raw_fn = fc.get('fine_n_components')
    fine_n_components = (list(raw_fn) if hasattr(raw_fn, '__iter__') else int(raw_fn)) if raw_fn is not None else None
    fit_and_save_pca(encoded_dir, output_dir, levels, n_components, key,
                     fine_levels=fine_levels, fine_n_components=fine_n_components)
    print("FINISHED")

if __name__ == '__main__' and 'ipykernel' not in __import__('sys').modules:
    fit_pca_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()